[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crunchdao/numinous/blob/master/challenge/numinous/examples/geopolitical_tracker.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/numinous/assets/banner.webp)

# Geopolitical Tracker — Using Numinous Indicia

This example shows how to use the **Numinous Indicia** API (free, no API key required) to
enrich predictions on geopolitical events with real-time OSINT signals from LiveUAMap.

The model:
1. Detects whether an event is geopolitical based on keywords in the title
2. Fetches live conflict signals from `indicia.numinouslabs.io`
3. Adjusts the market price based on signal intensity — more military activity → higher probability of escalation events

For non-geopolitical events it falls back to the market price.

## Setup

In [ ]:
%pip install crunch-numinous httpx

In [ ]:
from numinous.tracker import TrackerBase

## Numinous Indicia via the Gateway

Indicia signals are accessed through the gateway proxy (`SANDBOX_PROXY_URL`). Two endpoints are available — no API key needed:

| Gateway Endpoint | Returns |
|----------|--------|
| `POST /api/gateway/numinous-indicia/liveuamap` | Conflict/military signals (strikes, troop movements, explosions) |
| `POST /api/gateway/numinous-indicia/x-osint` | X/Twitter OSINT geopolitical signals |

Request body: `{"limit": N}` (optional `region` for liveuamap, `account` for x-osint).

Each signal has: `topic`, `category`, `signal` (text), `confidence`, `fact_status`, `timestamp`, `source_url`.

Categories include: `strike`, `explosion`, `troop_movement`, `ship`, `other`.

Let's see what's live right now:

In [ ]:
import os

import httpx

GATEWAY_BASE = os.getenv("SANDBOX_PROXY_URL")
USE_GATEWAY = GATEWAY_BASE is not None
if not USE_GATEWAY:
    GATEWAY_BASE = "https://indicia.numinouslabs.io"

print(f"Using {'gateway' if USE_GATEWAY else 'direct'}: {GATEWAY_BASE}")


def _indicia_request(path_gateway, path_direct, params):
    if USE_GATEWAY:
        resp = httpx.post(
            f"{GATEWAY_BASE}{path_gateway}",
            json=params,
            timeout=30.0,
        )
    else:
        resp = httpx.get(
            f"{GATEWAY_BASE}{path_direct}",
            params=params,
            timeout=30.0,
        )
    resp.raise_for_status()
    data = resp.json()
    return data.get("signals", data) if isinstance(data, dict) else data


signals = _indicia_request(
    "/api/gateway/numinous-indicia/liveuamap",
    "/signals/liveuamap",
    {"limit": 5},
)

for s in signals:
    print(f"[{s['category']:15s}] {s['signal'][:100]}")
    print(f"  confidence={s['confidence']}  fact_status={s['fact_status']}  url={s.get('source_url', '')}")
    print()

## The Model

The `GeopoliticalTracker` uses a simple heuristic:

1. Check if the event title contains geopolitical keywords (war, conflict, military, sanctions, etc.)
2. If yes, fetch recent Indicia signals and compute a "heat score" based on signal categories
3. Blend the market price with the heat score to shift the prediction

The intuition: when military activity is high, escalation-type events are more likely to resolve Yes.

In [ ]:
import logging
import re
from collections import Counter

from numinous.tracker import TrackerBase

logger = logging.getLogger(__name__)

GEO_KEYWORDS = re.compile(
    r"\b(war|conflict|military|troops|invasion|ceasefire|sanctions|nato|"
    r"ukraine|russia|missile|nuclear|escalat|strike|attack|offensive|"
    r"drone|weapon|defense|defence|army|frontline|territorial)\b",
    re.IGNORECASE,
)

CATEGORY_WEIGHTS = {
    "strike": 0.04,
    "explosion": 0.03,
    "troop_movement": 0.02,
    "ship": 0.01,
    "other": 0.005,
}

MAX_HEAT_SHIFT = 0.15


def fetch_indicia_signals(limit=20):
    return _indicia_request(
        "/api/gateway/numinous-indicia/liveuamap",
        "/signals/liveuamap",
        {"limit": limit},
    )


def compute_heat_score(signals):
    """Score 0..1 based on signal category mix and volume."""
    if not signals:
        return 0.0
    counts = Counter(s.get("category", "other") for s in signals)
    raw = sum(CATEGORY_WEIGHTS.get(cat, 0.005) * n for cat, n in counts.items())
    return min(raw, MAX_HEAT_SHIFT)


class GeopoliticalTracker(TrackerBase):
    """Market price + geopolitical signal adjustment.

    For geopolitical events, shifts the market price upward based on
    live conflict signal intensity from Numinous Indicia.
    For all other events, returns the market price as-is.
    """

    def _predict(self, subject):
        data = self._get_data(subject)
        if not isinstance(data, dict):
            return {"event_id": subject, "prediction": 0.5}

        event_id = data.get("event_id", subject)
        market_price = float(data.get("yes_price", 0.5))
        title = data.get("title", "")

        if not GEO_KEYWORDS.search(title):
            return {"event_id": event_id, "prediction": max(0.0, min(1.0, market_price))}

        signals = fetch_indicia_signals(limit=20)
        heat = compute_heat_score(signals)

        adjusted = market_price + heat
        prediction = max(0.0, min(1.0, adjusted))

        logger.info(
            "[GeoTracker] %s: market=%.3f heat=%.3f adjusted=%.3f signals=%d",
            event_id, market_price, heat, prediction, len(signals),
        )

        return {"event_id": event_id, "prediction": prediction}

## Test Locally

We test with a mix of geopolitical and non-geopolitical events.
The geopolitical events should get adjusted based on live Indicia signals,
while other events just follow the market.

In [ ]:
SAMPLE_EVENTS = [
    {
        "event_id": "evt-ukraine-ceasefire",
        "title": "Will there be a ceasefire in Ukraine before April?",
        "yes_price": 0.15,
        "cutoff": "2026-04-01T00:00:00Z",
        "source": "polymarket",
        "description": "Resolves Yes if an official ceasefire is declared.",
        "volume_24h": 800000.0,
        "metadata": {},
    },
    {
        "event_id": "evt-nato-expansion",
        "title": "Will NATO announce new military deployments to Eastern Europe?",
        "yes_price": 0.55,
        "cutoff": "2026-04-15T00:00:00Z",
        "source": "polymarket",
        "description": "",
        "volume_24h": 300000.0,
        "metadata": {},
    },
    {
        "event_id": "evt-btc-100k",
        "title": "Will BTC exceed $100k by end of March?",
        "yes_price": 0.65,
        "cutoff": "2026-04-01T00:00:00Z",
        "source": "polymarket",
        "description": "",
        "volume_24h": 150000.0,
        "metadata": {},
    },
    {
        "event_id": "evt-russia-sanctions",
        "title": "Will the EU impose new sanctions on Russia in Q2?",
        "yes_price": 0.70,
        "cutoff": "2026-06-30T00:00:00Z",
        "source": "polymarket",
        "description": "",
        "volume_24h": 200000.0,
        "metadata": {},
    },
]

OUTCOMES = {
    "evt-ukraine-ceasefire": 0,
    "evt-nato-expansion": 1,
    "evt-btc-100k": 1,
    "evt-russia-sanctions": 1,
}

model = GeopoliticalTracker()
total_brier = 0.0

print(f"{'Event':55s} {'Market':>7s} {'Pred':>6s} {'Out':>5s} {'Brier':>8s}")
print("-" * 85)

for event in SAMPLE_EVENTS:
    model.feed_update(event)
    result = model.predict(event["event_id"])

    p = max(0.01, min(0.99, result["prediction"]))
    o = OUTCOMES[event["event_id"]]
    brier = (p - o) ** 2
    total_brier += brier
    market = event["yes_price"]
    geo = "*" if GEO_KEYWORDS.search(event["title"]) else " "

    print(f"{geo}{event['title'][:54]:54s} {market:7.2f} {p:6.3f} {'Yes' if o else 'No':>5s} {brier:8.4f}")

print("-" * 85)
print(f" {'Average Brier':54s} {'':>7s} {'':>6s} {'':>5s} {total_brier / len(SAMPLE_EVENTS):8.4f}")
print(f" {'Baseline (always 0.5)':54s} {'':>7s} {'':>6s} {'':>5s} {'0.2500':>8s}")
print()
print("* = geopolitical event (adjusted with Indicia signals)")

## Inspect the Signals

Let's look at the raw signals and how the heat score is computed.

In [ ]:
signals = fetch_indicia_signals(limit=20)
heat = compute_heat_score(signals)

counts = Counter(s.get("category", "other") for s in signals)
print(f"Signal count: {len(signals)}")
print(f"Categories: {dict(counts)}")
print(f"Heat score: {heat:.4f} (max shift: {MAX_HEAT_SHIFT})")
print()

for s in signals[:5]:
    print(f"[{s['category']}] {s['signal'][:120]}")
    print(f"  confidence={s['confidence']}  source={s.get('source_url', 'n/a')}")
    print()

## Ideas for Improvement

This is a minimal example. Here's how you could make it better:

- **Match signals to events**: Use keyword overlap between signal text and event title to only count relevant signals
- **Direction detection**: Not all events resolve Yes on escalation — a ceasefire event resolves Yes when conflict *decreases*
- **Time decay**: Weight recent signals higher than older ones
- **Combine with an LLM**: Feed the signals into a prompt and ask the LLM to estimate P(Yes) given the evidence
- **Use both endpoints**: Combine LiveUAMap with X/Twitter OSINT for broader coverage
- **Category-specific strategies**: Different signal categories might indicate different things for different event types

## Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Deploy it in real condition

### >> https://hub.crunchdao.com/competitions/numinous/submit/notebook

The platform will find your `TrackerBase` subclass automatically.

![Download and Submit Notebook](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/download-and-submit-notebook-deployment.gif)